# Семинар 07. Протоколы и duck typing


## Цели

После семинара вы сможете:

- различать номинальную и структурную типизацию;
- описывать интерфейсы с ABC и typing.Protocol;
- понимать границы статической проверки типов во время выполнения.

## Перед началом

Повторите наследование, абстрактные классы и аннотации типов. Для демонстрации типов нужен nb-mypy.


**Динамическая типизация**
В языках со статической типизацией многие несоответствия типов обнаруживаются до выполнения программы — во время компиляции или статического анализа.

В Python тип связан с объектом и проверяется во время выполнения. Аннотации сами по себе не запрещают передать значение другого типа, поэтому возможны ошибка выполнения или неожиданное поведение. Статический анализатор помогает находить часть таких проблем заранее.

```python
class Rectangle:
    def __init__(self, width: int, height: int):
        self._width = width
        self._height = height

    def get_area(self):
        return self._width * self._height

if __name__ == "__main__":
    r = Rectangle(10, "tuesday")
    print(r.get_area())
```

Ну умножили вторник на ширину, что-то даже получилось, кто и что будет дальше делать с этим результатом и что получится - как повезет.

Как повысить надёжность типизации

`isinstance` проверяет тип объекта во время выполнения с учётом наследования. Такие проверки полезны на границах системы, но повсеместное их добавление увеличивает объём кода и не заменяет статический анализ.


In [ ]:
import abc

class Flyable(abc.ABC):
    @abc.abstractmethod
    def fly(self):
        pass

class Junkie(abc.ABC):
    @abc.abstractmethod
    def fly(self):
        pass

class Bird(Flyable):
    def fly(self):
        print("I'm flying")

class Hippie(Junkie):
    def fly(self):
        print("I'm flying high")

class FlyableManager:
    def __init__(self):
        self.flyables: list[Flyable] = [] # список объектов, которые могут летать

    def add_flyable(self, flyable: Flyable):
        # if not isinstance(flyable, Flyable):
        #     # здесь мы проверяем, что переданный объект является Flyable
        #     raise TypeError("FlyableManager can only add Flyable objects")
        self.flyables.append(flyable)
    
    def fly(self):
        for flyable in self.flyables:
            flyable.fly()

manager = FlyableManager()
manager.add_flyable(Bird())
# manager.add_flyable("tuesday")
# Несмотря на то, что Hippie тоже умеет летать, этот класс не наследуется от Flyable.
manager.add_flyable(Hippie())
manager.fly()

### Протоколы и duck typing
Протокол описывает требуемый набор методов. Явно наследоваться от него необязательно: класс соответствует протоколу, если предоставляет совместимые методы. Если объект ходит как утка и крякает как утка, его можно использовать там, где нужна такая «утка».

In [ ]:
%load_ext nb_mypy
from typing import Protocol

class Flyable(Protocol):
    def fly(self):
        pass

class Bird():
    def fly(self):
        print("I'm flying")

class Hippie:
    def fly(self):
        print("I'm flying high")

class ParametrizedFlyer():
    def fly(self, speed: int):
        print(f"I'm flying at {speed} knots")

class FlyableManager:
    def __init__(self):
        self.flyables = []

    def add_flyable(self, flyable: Flyable):
        # Из коробки не заработает, потому что протоколы для isinstance нужно дополнительно размечать
        # if not isinstance(flyable, Flyable):
        #     # здесь мы проверяем, что переданный объект является Flyable
        #     raise TypeError("FlyableManager can only add Flyable objects")
        self.flyables.append(flyable)
    
    def fly(self):
        for flyable in self.flyables:
            flyable.fly()

manager = FlyableManager()
manager.add_flyable(Bird())
# Hippie явно не наследуется от Flyable, но соответствует протоколу
manager.add_flyable(Hippie())

manager.fly()
# а вот это уже не работает
# manager.add_flyable(ParametrizedFlyer())
# число тоже не соответствует протоколу:
# manager.add_flyable(1)


## Задание 1. Круглые скобки (1 балл)

Строка состоит только из `(` и `)`. Определите, является ли она правильной скобочной последовательностью.

```python
def is_valid_parentheses(text: str) -> bool:
    ...
```

**Примеры:** `""`, `"()"` и `"(())()"` валидны; `")("` и `"(()"` невалидны.

**Критерии проверки:** `O(N)` по времени, `O(1)` по дополнительной памяти; промежуточный баланс никогда не становится отрицательным и в конце равен нулю.


## Задание 2. Несколько видов скобок (2 балла)

Строка состоит из символов `()[]{}`. Закрывающая скобка должна соответствовать последней незакрытой скобке.

```python
def is_valid_brackets(text: str) -> bool:
    ...
```

**Примеры:** `"[]()"` и `"{[()]}"` валидны; `"[(])"` и `"{{}"` невалидны.

**Критерии проверки:** `O(N)` по времени и `O(N)` по памяти; решение корректно обрабатывает пустую строку и закрывающую скобку в начале.
